# Notebook 10: Hybrid Retrieval (BM25 + Dense FAISS via RRF)

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
In **Phase 3**, we implemented lexical search (**BM25**), which excels at exact token matching (restaurant names, street numbers, specific dishes).
In **Phase 4**, we implemented dense vector search (**FAISS + Multilingual-E5**), which excels at semantic similarity, intent, vibes, and cross-lingual queries.

However, both systems fail in distinct ways:
- **BM25 fails** when users search using synonyms, vibes, or descriptions not present verbatim in the text (*vocabulary mismatch*).
- **Vector search fails** when users search for specific proper nouns, addresses, or telephone numbers, drifting towards semantically related places instead of the exact target (*semantic drift*).

**Hybrid Retrieval** solves this by fusing lexical and dense search.

### Core Objectives:
1. Implement and test `HybridRetriever` using **Reciprocal Rank Fusion (RRF)**:
   $$\text{RRF}(d) = \sum_{m \in \{\text{BM25}, \text{Vector}\}} \frac{w_m}{k + r_m(d)}$$
2. Compare BM25, Vector Search, and Hybrid across diverse real-world query types.
3. Perform **Weight Tuning & Grid Search** across weight ratios: $1.0/0.0, 0.8/0.2, 0.5/0.5, 0.4/0.6, 0.2/0.8, 0.0/1.0$.
4. Compare **RRF (Rank-Based)** vs. **Min-Max Linear Score Fusion**.
5. Deconstruct hybrid score contributions term-by-term using `explain_fusion()`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.preprocessing.text_preprocessing import TextPreprocessor
from src.indexing.inverted_index import InvertedIndex
from src.retrieval.bm25 import BM25Retriever
from src.embeddings.vector_store import VectorStore
from src.embeddings.embedding_model import EmbeddingModel
from src.retrieval.vector_search import VectorSearcher
from src.retrieval.hybrid import HybridRetriever

print(f"Project root resolved: {PROJECT_ROOT}")

## 1. Load Data, Inverted Index, and FAISS Vector Store

In [ ]:
# Load 698 restaurants
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
docs = converter.convert_all(records)
corpus = converter.get_corpus()

# 1. Initialize Lexical BM25 Retriever
preprocessor = TextPreprocessor()
tokenized_corpus = preprocessor.preprocess_documents(corpus)
index = InvertedIndex()
index.build(tokenized_corpus)
bm25 = BM25Retriever(index, preprocessor)

# 2. Initialize Dense FAISS Vector Searcher
idx_path = PROJECT_ROOT / "vector_db" / "faiss_index.bin"
meta_path = PROJECT_ROOT / "vector_db" / "index_metadata.json"
vector_store = VectorStore.load(idx_path, meta_path)
embedding_model = EmbeddingModel()
vector_searcher = VectorSearcher(vector_store=vector_store, embedding_model=embedding_model)

# 3. Initialize Hybrid Retriever
hybrid = HybridRetriever(
    bm25_retriever=bm25,
    vector_searcher=vector_searcher,
    bm25_weight=0.5,
    vector_weight=0.5,
    rrf_k=60,
)

print("HybridRetriever successfully initialized with BM25 and FAISS Vector Search!")

## 2. Multi-Engine Comparison Across 4 Query Archetypes

We compare **BM25 alone**, **Vector Search alone**, and **Hybrid (RRF)** on 4 distinct queries:
1. **Exact Entity & Street Query**: *"Malis Restaurant Street 308"*
2. **Specific Dish & District**: *"nom banh chok in boeng keng kang"*
3. **Atmosphere / Concept / Vibe**: *"romantic cozy dinner spot with wine"*
4. **Cross-Lingual Khmer Script**: *"ហាងកាហ្វេស្ងប់ស្ងាត់"* (quiet coffee shop)

In [ ]:
queries = [
    ("Exact Entity / Address", "Malis Restaurant Street 308"),
    ("Dish & District", "nom banh chok in boeng keng kang"),
    ("Concept / Vibe", "romantic cozy dinner spot with wine"),
    ("Khmer Query", "ហាងកាហ្វេស្ងប់ស្ងាត់"),
]

for category, q in queries:
    print(f"\n{'='*80}\n[{category.upper()}] Query: '{q}'\n{'='*80}")
    
    bm_res = bm25.search(q, top_k=3)
    vec_res = vector_searcher.search(q, top_k=3)
    hyb_res = hybrid.search(q, top_k=3)
    
    print("\n--- 1. BM25 (Lexical) ---")
    if not bm_res:
        print("  [No results found]")
    for r in bm_res:
        name = converter.get_record(r["doc_id"])["name"]
        print(f"  [{r['rank']}] {name} (BM25: {r['score']:.4f})")
        
    print("\n--- 2. Vector Search (FAISS Dense) ---")
    for r in vec_res:
        name = converter.get_record(r["doc_id"])["name"]
        print(f"  [{r['rank']}] {name} (Cosine: {r['score']:.4f})")
        
    print("\n--- 3. Hybrid Search (RRF 50/50) ---")
    for r in hyb_res:
        name = converter.get_record(r["doc_id"])["name"]
        bm_info = f"BM25 rank {r['bm25_rank']}" if r['bm25_rank'] else "No BM25"
        vec_info = f"Vec rank {r['vector_rank']}" if r['vector_rank'] else "No Vec"
        print(f"  [{r['rank']}] {name} | RRF: {r['score']:.5f} ({bm_info}, {vec_info})")

## 3. Weight Tuning & Grid Search

Now let us systematically test different weight combinations $(w_{\text{bm25}} : w_{\text{vector}})$:
- $1.0 : 0.0$ (Pure BM25)
- $0.8 : 0.2$ (BM25 heavy)
- $0.6 : 0.4$ (Slight BM25 tilt)
- $0.5 : 0.5$ (Balanced)
- $0.4 : 0.6$ (Slight Vector tilt)
- $0.2 : 0.8$ (Vector heavy)
- $0.0 : 1.0$ (Pure Vector)

We observe how the candidate ranking shifts for a multi-aspect query: *"wood fired pizza and craft beer in tonle bassac"*.

In [ ]:
sweep_query = "wood fired pizza and craft beer in tonle bassac"
weights_grid = [
    (1.0, 0.0, "100% BM25"),
    (0.8, 0.2, "80% BM / 20% Vec"),
    (0.6, 0.4, "60% BM / 40% Vec"),
    (0.5, 0.5, "50% BM / 50% Vec (Balanced)"),
    (0.4, 0.6, "40% BM / 60% Vec"),
    (0.2, 0.8, "20% BM / 80% Vec"),
    (0.0, 1.0, "100% Vector"),
]

print(f"Sweep Query: '{sweep_query}'\n" + "="*75)
rows = []

for w_bm, w_vec, label in weights_grid:
    res = hybrid.search(sweep_query, top_k=3, bm25_weight=w_bm, vector_weight=w_vec)
    top_names = [converter.get_record(r["doc_id"])["name"] for r in res]
    rows.append({
        "Configuration": label,
        "Top 1": top_names[0] if len(top_names) > 0 else "None",
        "Top 2": top_names[1] if len(top_names) > 1 else "None",
        "Top 3": top_names[2] if len(top_names) > 2 else "None",
    })

sweep_df = pd.DataFrame(rows)
sweep_df

### Key Observation:
- At $1.0 : 0.0$ (pure BM25), places with exact string matches for *'pizza'* or *'beer'* dominate.
- At $0.0 : 1.0$ (pure Vector), places associated with Italian dining or craft brew atmospheres appear.
- At balanced weights ($0.4:0.6$ or $0.5:0.5$), the engine finds places that **simultaneously** satisfy the exact location keyword (*'Tonle Bassac'*) AND the culinary concept (*'wood fired pizza & craft beer'*).

## 4. Comparing Fusion Methods: RRF vs. Min-Max Linear Score Fusion

We now compare **Reciprocal Rank Fusion (RRF)** against **Min-Max Normalized Linear Fusion** on the query: *"cheap street breakfast noodles"*.

In [ ]:
compare_query = "cheap street breakfast noodles"

rrf_results = hybrid.search(compare_query, top_k=3, fusion_method="rrf")
linear_results = hybrid.search(compare_query, top_k=3, fusion_method="linear")

print(f"QUERY: '{compare_query}'\n" + "="*75)
print("--- Reciprocal Rank Fusion (RRF) ---")
for r in rrf_results:
    name = converter.get_record(r["doc_id"])["name"]
    print(f"  [{r['rank']}] {name:35s} | Score: {r['score']:.5f} (BM25 Rank: {r['bm25_rank']}, Vec Rank: {r['vector_rank']})")

print("\n--- Normalized Linear Score Fusion ---")
for r in linear_results:
    name = converter.get_record(r["doc_id"])["name"]
    print(f"  [{r['rank']}] {name:35s} | Score: {r['score']:.5f} (BM25 Rank: {r['bm25_rank']}, Vec Rank: {r['vector_rank']})")

## 5. Deconstructing Scores via `explain_fusion()`

`explain_fusion()` provides a complete audit trail showing exactly how a restaurant achieved its final hybrid rank.

In [ ]:
sample_doc_id = rrf_results[0]["doc_id"]
sample_name = converter.get_record(sample_doc_id)["name"]

explanation = hybrid.explain_fusion(compare_query, doc_id=sample_doc_id, fusion_method="rrf")

print(f"Score Explanation for '{sample_name}' [ID: {sample_doc_id}]:")
print("-" * 70)
for k, v in explanation.items():
    print(f"  {k:25s}: {v}")

## 6. Summary and Takeaways

1. **Reciprocal Rank Fusion (RRF)** provides robust, outlier-proof combination of lexical and dense retrievers without requiring score normalization.
2. **Consensus Prioritization**: Documents appearing in the top ranks of both BM25 and Vector search receive exponential boosts over single-engine candidates.
3. **Weight Tuning Flexibility**: While $0.5 : 0.5$ provides a clean, balanced baseline, tuning to $0.4 \text{ BM25} : 0.6 \text{ Vector}$ provides the ideal trade-off for short restaurant metadata containing both exact addresses and rich cuisine concepts.
4. **Next Step (Phase 6)**: In Phase 6, we will systematically benchmark precision, recall, and MRR using a labeled evaluation test set!